# Blind Search Algorithms Lab

**Student Name:** [Your Name]  
**Roll Number:** [Your Roll Number]  
**Date:** [Date]  

---

## Task 1: File Reading and Validation (15 points)

Implement the `read_grid(filename)` function that:
- Reads a grid from a text file
- Validates the grid (size, exactly one S, at least one G, valid characters)
- Returns: grid (2D list), start position (row, col), list of goal positions

In [32]:
def read_grid(filename):
    """
    Read and validate grid from file.
    
    Args:
        filename: Path to grid file
        
    Returns:
        tuple: (grid, start_pos, goal_positions)
        - grid: 2D list
        - start_pos: (row, col) tuple
        - goal_positions: list of (row, col) tuples
        
    Raises:
        ValueError: If grid is invalid
    """

    valid_chars = ['0', '1', '2', 'S', 'G']
    start_flag, end_flag = False , False
    row_size = 0
    
    grid = []
    start_pos = []
    goal_pos = []

    empty_res = ([], (-1), [])
    
    with open(filename, 'r') as file:
        i = 0
        for line in file:
            row = (line.strip().split(' '))
            print(row)
            # grid size check
            if len(row) < 3:
                print("Grid is not 3 x 3")
                return empty_res
                
            if not row_size:
                row_size = len(row) # putting row size first time
            # consistant row size check
            if len(row) != row_size:
                print(f"Grid size is not cosistent in row# {i}")
                return empty_res
                
            j = 0 
            for cell in row:
                cell = cell.upper()
                if cell not in valid_chars:
                    print(f"Invalid character '{cell}' found at index: {i},{j}")
                    return empty_res

                if cell == 'S' and start_flag:
                    print("Multiple start position in grid")
                    return
                elif cell == 'S' and not start_flag:
                    start_pos = (i, j)
                    start_flag = True # marking first start as found
                
                if cell == 'G':
                    goal_pos.append((i,j))
                    end_flag = True # marking end found
                    
                j += 1



            grid.append(row)
            i += 1
        if not end_flag:
            print("No Goal is found in grid")
            return empty_res
        if not start_flag:
            print("No start is found in grid")
        return tuple([grid, start_pos, goal_pos])
        
        
    pass

# Test your function
grid, start, goals = read_grid('grid.txt')
print(f"Start: {start}")
print(f"Goals: {goals}")

['1', '0', 'S', '2', '1', '0', '0']
['1', '1', '0', '0', '2', '1', '1']
['0', '1', '0', '1', '0', '2', '0']
['1', '1', '2', '1', '1', '0', '1']
['0', '1', '0', '1', '2', '0', 'G']
Start: (0, 2)
Goals: [(4, 6)]


## Helper Function: Get Neighbors

Implement a function to get valid neighbors in the order: Up, Right, Down, Left

In [33]:
def is_valid_pos(grid, row, col):
    if row < 0 or col < 0 or row >= len(grid) or col >= len(grid[0]):
        #print(f"Not Valid row {row} and col{col}")
        return False

    return True

def is_wall(grid, row, col):
    if grid[row][col] == '1':
        return True
    return False

def is_mud(grid, row, col):
    if grid[row][col] == '2':
        return True
    return False
       


def get_neighbors(grid, row, col):
    """
    Get valid neighbors of a cell in order: Up, Right, Down, Left.
    
    Args:
        grid: 2D list representing the maze
        row: Current row
        col: Current column
        
    Returns:
        list of tuples: [(row, col, cost), ...] for valid neighbors
        - cost = 1 for normal cells (0, S, G)
        - cost = 3 for mud cells (2)
    """
    neighbors = []
    if not is_valid_pos(grid, row, col):
        return
    positions = [(-1, 0), # up
                (0, 1), # right
                (1, 0), # down
                (0, -1)] # left]
    
    for x,y in positions:
        if is_valid_pos(grid, row + x, col + y) and not is_wall(grid, row + x, col + y):
            if is_mud(grid, row + x, col + y):
                neighbors.append((row + x, col + y, 3))
            else:
                neighbors.append((row + x, col + y, 1))
    
    return neighbors
    pass

#Test
print(get_neighbors(grid, 1, 3))

[(0, 3, 3), (1, 4, 3), (1, 2, 1)]


## Task 2A: Depth-First Search (15 points)

Implement DFS with:
- Stack or recursion
- Maximum depth limit of 50
- Neighbor order: Up, Right, Down, Left
- Stop at first goal

In [34]:


def dfs(grid, start, goals, DEPTH_LIMIT = 50):
    """
    Depth-First Search with depth limit of 50.
    
    Args:
        grid: 2D list representing the maze
        start: (row, col) tuple for start position
        goals: list of (row, col) tuples for goal positions
        
    Returns:
        dict: {
            'path': list of (row, col) from start to goal or None,
            'goal_reached': (row, col) of goal or None,
            'cost': total path cost,
            'nodes_expanded': number of nodes explored
        }
    """
    path = None
    goal_reached = None
    total_cost = 0
    nodes_expanded = 0
    # what should be in closeList?
    # dic of (row, col): (cost, parent)

    visited = set()
    closeList = {}
    openList = []; # will be a stack for dfs
    # ( (row, col), cost, (parentRow, parentCol), depth ) 
    openList.append( ((start[0], start[1]), 0, None, 0) )
    visited.add(start)
    
    while openList:
        ## expanding frontier from queue
        curr = openList.pop()
        curr_x, curr_y = curr[0]
        curr_cost, curr_parent, curr_depth = curr[1], curr[2], curr[3]
        neighbors = get_neighbors(grid, curr_x, curr_y)

        ## depth check
        if(curr_depth >= DEPTH_LIMIT):
            continue
        for neighbor in neighbors:
            
            x, y, n_cost = neighbor[0], neighbor[1], neighbor[2]
            
            ## goal is found
            if (x, y) in goals: 
                closeList[(x, y)] =  (n_cost, (curr_x, curr_y))
                goal_reached = (x, y) # pos of goal founded
                openList.clear() # marking openList empty so that outer loop can be breaked
                break

            ## we have check neighbor is visited or not
            if (x, y) in visited:
                continue # already visited
            
            ## add neighbor to open list
            openList.append( ((x, y), n_cost, (curr_x, curr_y), curr_depth + 1) ) 
            visited.add((x, y))
            
        closeList[ (curr_x, curr_y) ] = (curr_cost, curr_parent)
        nodes_expanded += 1
        
    # print(closeList)
    
    if goal_reached:
        path = []
        path.append(goal_reached)
        curr = goal_reached

        while(curr != None and curr != start ):
            cost, parent = closeList[curr]

            path.append(parent)
            total_cost += cost
            
            curr = parent
            
        # reversing the path as started from goal
        path = path[::-1]
            
    return {
        'path' : path,
        'goal_reached' : goal_reached,
        'cost' : total_cost,
        'nodes_expanded' : nodes_expanded
    }
# Test your implementation
result = dfs(grid, start, goals)
print(f"DFS Result: {result}")

DFS Result: {'path': [(0, 2), (1, 2), (1, 3), (1, 4), (2, 4), (2, 5), (3, 5), (4, 5), (4, 6)], 'goal_reached': (4, 6), 'cost': 12, 'nodes_expanded': 12}


## Task 2B: Breadth-First Search (15 points)

Implement BFS with:
- Queue (FIFO)
- Same neighbor order as DFS
- Stop at first goal

In [35]:
from collections import deque

def bfs(grid, start, goals):
    """
    Breadth-First Search.
    
    Args:
        grid: 2D list representing the maze
        start: (row, col) tuple for start position
        goals: list of (row, col) tuples for goal positions
        
    Returns:
        dict: Same format as DFS
    """
    path = None
    goal_reached = None
    total_cost = 0
    nodes_expanded = 0
    # what should be in closeList?
    # dic of (row, col): (cost, parent)

    visited = set()
    closeList = {}
    openList = deque();
    # ( (row, col), cost, (parentRow, parentCol) ) 
    openList.append( ((start[0], start[1]), 0, None) )
    visited.add(start)
    
    

    
    while openList:
        ## expanding frontier from queue
        curr = openList.popleft()
        curr_x, curr_y = curr[0]
        curr_cost, curr_parent = curr[1], curr[2]
        neighbors = get_neighbors(grid, curr_x, curr_y)
        
        for neighbor in neighbors:
            
            x, y, n_cost = neighbor[0], neighbor[1], neighbor[2]
            
            ## goal is found
            if (x, y) in goals: 
                closeList[(x, y)] =  (n_cost, (curr_x, curr_y))
                goal_reached = (x, y) # pos of goal founded
                openList.clear() # marking openList empty so that outer loop can be breaked
                break

            ## we have check neighbor is visited or not
            if (x, y) in visited:
                continue # already visited
            
            ## add neighbor to open list
            openList.append( ((x, y), n_cost, (curr_x, curr_y)) )
            visited.add((x, y))
            
        closeList[ (curr_x, curr_y) ] = (curr_cost, curr_parent)
        nodes_expanded += 1
        
    # print(closeList)
    
    if goal_reached:
        path = []
        path.append(goal_reached)
        curr = goal_reached

        while(curr != None and curr != start ):
            cost, parent = closeList[curr]

            path.append(parent)
            total_cost += cost
            
            curr = parent
            
        # reversing the path as started from goal
        path = path[::-1]
            
    return {
        'path' : path,
        'goal_reached' : goal_reached,
        'cost' : total_cost,
        'nodes_expanded' : nodes_expanded
    }

# Test your implementation
result = bfs(grid, start, goals)
print(f"BFS Result: {result}")

BFS Result: {'path': [(0, 2), (0, 3), (1, 3), (1, 4), (2, 4), (2, 5), (3, 5), (4, 5), (4, 6)], 'goal_reached': (4, 6), 'cost': 14, 'nodes_expanded': 14}


## Task 2C: Uniform Cost Search (20 points)

Implement UCS with:
- Priority queue (min-heap)
- Cost: normal cells = 1, mud = 3
- Always expand lowest cost node

In [39]:
import heapq

def ucs(grid, start, goals):
    """
    Uniform Cost Search.
    
    Args:
        grid: 2D list representing the maze
        start: (row, col) tuple for start position
        goals: list of (row, col) tuples for goal positions
        
    Returns:
        dict: Same format as DFS
    """

    path = None
    goal_reached = None
    total_cost = 0
    nodes_expanded = 0
    # what should be in closeList?
    # dic of (row, col): (cost, parent)

    visited = set()
    closeList = {}
    openList = [];
    # ( cost, (row, col),  (parentRow, parentCol) ) 
    heapq.heappush(openList, (0, (start[0], start[1]), None) )
    
    

    
    while openList:
        ## expanding frontier from queue
        (curr_cost, curr, curr_parent) = heapq.heappop(openList)

        ## we have check wheter its already came out of heap
        if curr in visited:
            continue # already visited
            
        ## goal is found
        if curr in goals: 
            # curr_cost + neighbor_cost beacause commulatitive
            closeList[curr] =  (curr_cost, curr_parent) 
            goal_reached = curr # pos of goal founded
            openList.clear() # marking openList empty so that outer loop can be breaked
            total_cost = curr_cost # as commulative cost
            break
        visited.add(curr)
        neighbors = get_neighbors(grid, curr[0], curr[1])
        
        for neighbor in neighbors:
            x, y, n_cost = neighbor[0], neighbor[1], neighbor[2]
            
            if (x, y) not in visited:
                ## add neighbor to open list
                heapq.heappush( openList, 
                               ( curr_cost + n_cost, (x, y), curr )
                )
            
        closeList[curr] = (curr_cost, curr_parent)
        nodes_expanded += 1
        
    # print(closeList)
    
    if goal_reached:
        path = []
        path.append(goal_reached)
        curr = goal_reached

        while(curr != None and curr != start ):
            cost, parent = closeList[curr]

            path.append(parent)
            
            curr = parent
            
        # reversing the path as started from goal
        path = path[::-1]
            
    return {
        'path' : path,
        'goal_reached' : goal_reached,
        'cost' : total_cost,
        'nodes_expanded' : nodes_expanded
    }
    
    # TODO: Implement UCS
    # Use heapq for priority queue
    # Format: heapq.heappush(queue, (cost, node))
    pass

# Test your implementation
result = ucs(grid, start, goals)
print(f"UCS Result: {result}")

UCS Result: {'path': [(0, 2), (1, 2), (1, 3), (1, 4), (2, 4), (2, 5), (3, 5), (4, 5), (4, 6)], 'goal_reached': (4, 6), 'cost': 12, 'nodes_expanded': 14}


## Task 2D: Iterative Deepening Search (20 points)

Implement IDS with:
- Start with depth limit = 0
- Increment by 2 each iteration (0, 2, 4, 6, 8...)
- Maximum depth limit = 30
- Each iteration is a depth-limited DFS

In [24]:
def ids(grid, start, goals):
    """
    Iterative Deepening Search.
    Increment depth limit by 2 each iteration (0, 2, 4, 6...).
    Maximum depth limit is 30.
    
    Args:
        grid: 2D list representing the maze
        start: (row, col) tuple for start position
        goals: list of (row, col) tuples for goal positions
        
    Returns:
        dict: Same format as DFS
    """
    # reusing dfs function with DEPTH_LIMIT passed to it as parameter
    result = {}
    for i in range(0, 31):
        result = dfs(grid, start, goals, i)
        if result['goal_reached'] is not None:
            break

    
    return result

# Test your implementation
result = ids(grid, start, goals)
print(f"IDS Result: {result}")

IDS Result: {'path': [(0, 2), (1, 2), (1, 3), (1, 4), (2, 4), (2, 5), (3, 5), (4, 5), (4, 6)], 'goal_reached': (4, 6), 'cost': 12, 'nodes_expanded': 12}


## Task 3: Comparison Function (15 points)

Compare all four algorithms and display results in a formatted table

In [52]:
def compare_algorithms(grid_file):
    """
    Run all four algorithms and compare results.
    
    Args:
        grid_file: Path to grid file
    """
    print(f"\nAlgorithm Performance Comparison")
    print("=" * 80)
    print(f"Grid: {grid_file}\n")
    
    # Read grid
    grid, start, goals = read_grid(grid_file)
    
    # Run algorithms
    dfs_result = dfs(grid, start, goals)
    bfs_result = bfs(grid, start, goals)
    ucs_result = ucs(grid, start, goals)
    ids_result = ids(grid, start, goals)

    results = {
        "DFS": dfs_result,
        "BFS": bfs_result,
        "UCS": ucs_result,
        "IDS": ids_result
    }
    
    # Display table
    print(f"{'Algorithm':<12} | {'Goal Reached':<15} | {'Path Cost':<10} | {'Nodes Expanded':<15}")
    print("-" * 80)

    for algo_name, result in results.items():
        goal = result['goal_reached'] 
        cost = result['cost'] 
        nodes = result['nodes_expanded']
        
        print(f"{algo_name:<12} | {str(goal):<15} | {str(cost):<10} | {str(nodes):<15}")

    # TODO: Print results for each algorithm
    # Format: print(f"{algo:<12} | {goal:<15} | {cost:<10} | {nodes:<15}")
    
    print("\nNotes:")
    # TODO: Add observations about which algorithm found which goal and why

## Test Cases

Create test grid files and run the comparison

In [42]:
# Create sample grid 1 file
sample_grid1 = """1 0 S 2 1 0 0
1 1 0 0 2 1 1
0 1 0 1 0 2 0
1 1 2 1 1 0 1
0 1 0 1 2 0 G"""

with open('sample_grid1.txt', 'w') as f:
    f.write(sample_grid1)

# Create sample grid 2 file (multiple goals)
sample_grid2 = """S 0 0 G 1 0 0
0 1 0 0 2 1 1
0 1 0 1 0 2 0
1 1 2 1 1 0 G
G 0 2 0 0 0 0"""

with open('sample_grid2.txt', 'w') as f:
    f.write(sample_grid2)

print("Test grid files created!")

Test grid files created!


In [53]:
# Run comparison on sample grid 1
compare_algorithms('sample_grid1.txt')


Algorithm Performance Comparison
Grid: sample_grid1.txt

['1', '0', 'S', '2', '1', '0', '0']
['1', '1', '0', '0', '2', '1', '1']
['0', '1', '0', '1', '0', '2', '0']
['1', '1', '2', '1', '1', '0', '1']
['0', '1', '0', '1', '2', '0', 'G']
Algorithm    | Goal Reached    | Path Cost  | Nodes Expanded 
--------------------------------------------------------------------------------
DFS          | (4, 6)          | 12         | 12             
BFS          | (4, 6)          | 14         | 14             
UCS          | (4, 6)          | 12         | 14             
IDS          | (4, 6)          | 12         | 12             

Notes:


In [54]:
# Run comparison on sample grid 2 (multiple goals)
compare_algorithms('sample_grid2.txt')


Algorithm Performance Comparison
Grid: sample_grid2.txt

['S', '0', '0', 'G', '1', '0', '0']
['0', '1', '0', '0', '2', '1', '1']
['0', '1', '0', '1', '0', '2', '0']
['1', '1', '2', '1', '1', '0', 'G']
['G', '0', '2', '0', '0', '0', '0']
Algorithm    | Goal Reached    | Path Cost  | Nodes Expanded 
--------------------------------------------------------------------------------
DFS          | (0, 3)          | 3          | 5              
BFS          | (0, 3)          | 3          | 4              
UCS          | (0, 3)          | 3          | 5              
IDS          | (0, 3)          | 3          | 5              

Notes:


---

## Submission Checklist

Before submitting, ensure:
- [x] All functions are implemented and tested
- [x] Code runs without errors
- [x] Comparison output is displayed correctly
- [x] Notebook is renamed to your roll number (e.g., l234212.ipynb)
- [x] File is saved as .ipynb (NOT zipped)

**Good Luck!**